# 01 — Topic Modeling: Alt Media vs Mainstream

This notebook trains separate BERTopic models on the **alternative media** corpus and the **mainstream** (Tagesschau) corpus.

## Methodological Grounding

We use **BERTopic** (Grootendorst, 2022) rather than LDA because:
1. BERTopic uses contextual sentence embeddings (`paraphrase-multilingual-MiniLM-L12-v2`) instead of bag-of-words, capturing semantic similarity that LDA misses in short news texts.
2. Topic count is determined dynamically by HDBSCAN clustering rather than fixed a priori.
3. c-TF-IDF topic representations are more interpretable than LDA word distributions for German-language corpora with compound words.

**Reference**: Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure. *arXiv:2203.05794*.

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "agenda_distortion" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import importlib
import pandas as pd
from IPython.display import display

from bertopic_config import BERTopicConfig
from bertopic_pipeline import (
    prepare_documents,
    build_topic_model,
    run_bertopic_pipeline,
)
import merged_outlets_analysis as moa

moa = importlib.reload(moa)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

## Corpus Inspection

Load all outlet dataframes using the existing loaders from `merged_outlets_analysis.py`, then split into **mainstream** (Tagesschau) and **alt media** (all others).

In [ ]:
OUTLET_SPECS = moa.OUTLET_SPECS
ALT_MEDIA_OUTLET_KEYS = moa.ALT_MEDIA_OUTLET_KEYS

# Load and prepare all outlet documents using the existing pipeline
prepared_by_outlet = moa.load_all_prepared_documents(PROJECT_ROOT)

# Split into mainstream vs alternative
mainstream_prepared = prepared_by_outlet["tagesschau"]
alt_parts = [prepared_by_outlet[key] for key in ALT_MEDIA_OUTLET_KEYS]
alt_prepared = pd.concat(alt_parts, ignore_index=True)

print(f"Mainstream (Tagesschau): {len(mainstream_prepared):,} prepared documents")
print(f"Alt media (combined):    {len(alt_prepared):,} prepared documents")
print()

# Per-outlet breakdown
for key, df in prepared_by_outlet.items():
    label = OUTLET_SPECS[key].label
    category = "Mainstream" if key == "tagesschau" else "Alt media"
    print(f"  {label:<22} {len(df):>6,}  ({category})")

# Imbalance warning
ratio = max(len(mainstream_prepared), len(alt_prepared)) / min(len(mainstream_prepared), len(alt_prepared))
print(f"\nCorpus size ratio: {ratio:.1f}x")
if ratio > 2.0:
    print(
        "⚠ WARNING: Corpus imbalance > 2x detected. "
        "All prevalence metrics MUST be normalized. "
        "Bootstrap CIs are required in notebook 02."
    )
else:
    print("✓ Corpus sizes are reasonably balanced.")

## Train BERTopic on Alt Media Corpus

Uses `calculate_probabilities=True` so we get per-document topic probability distributions for downstream KPI computation.

In [ ]:
from bertopic_pipeline import build_embedding_model

# Shared embedding model — same instance for both corpora to ensure comparability
embedding_model = build_embedding_model(BERTopicConfig())

alt_config = BERTopicConfig(
    calculate_probabilities=True,
    hdbscan_min_cluster_size=20,
    hdbscan_min_samples=5,
    umap_n_neighbors=25,
    umap_min_dist=0.0,
    random_state=RANDOM_STATE,
)

alt_docs = alt_prepared["document"].tolist()
alt_topic_model = build_topic_model(alt_config, embedding_model=embedding_model)
alt_topics, alt_probs = alt_topic_model.fit_transform(alt_docs)

alt_topic_info = alt_topic_model.get_topic_info()
print(f"Alt media: {len(alt_topic_info)} topics (including outlier topic -1)")
print(f"Outlier rate: {(pd.Series(alt_topics) == -1).mean():.1%}")

## Train BERTopic on Mainstream Corpus

Same config settings for comparability.

In [ ]:
ms_config = BERTopicConfig(
    calculate_probabilities=True,
    hdbscan_min_cluster_size=20,
    hdbscan_min_samples=5,
    umap_n_neighbors=25,
    umap_min_dist=0.0,
    random_state=RANDOM_STATE,
)

ms_docs = mainstream_prepared["document"].tolist()
ms_topic_model = build_topic_model(ms_config, embedding_model=embedding_model)
ms_topics, ms_probs = ms_topic_model.fit_transform(ms_docs)

ms_topic_info = ms_topic_model.get_topic_info()
print(f"Mainstream: {len(ms_topic_info)} topics (including outlier topic -1)")
print(f"Outlier rate: {(pd.Series(ms_topics) == -1).mean():.1%}")

## Sanity Check: Top 10 Words for Top 10 Topics

Inspect topic coherence before proceeding. If topics look like noise, tune `hdbscan_min_cluster_size` above.

In [ ]:
def print_top_topics(model, label, n_topics=10, n_words=10):
    """Print the top words for the top-N topics of a BERTopic model."""
    topic_info = model.get_topic_info()
    non_outlier = topic_info.loc[topic_info["Topic"] != -1].head(n_topics)
    print(f"\n{'='*60}")
    print(f"  {label} — Top {n_topics} Topics")
    print(f"{'='*60}")
    for _, row in non_outlier.iterrows():
        topic_id = row["Topic"]
        words = model.get_topic(topic_id)
        word_str = ", ".join(w for w, _ in words[:n_words])
        print(f"  Topic {topic_id:>3} ({row['Count']:>5} docs): {word_str}")


print_top_topics(alt_topic_model, "Alt Media")
print_top_topics(ms_topic_model, "Mainstream (Tagesschau)")

### Decision Point

**Do these topics make intuitive sense?**
- Are there garbage/noise topics?
- Is the outlier rate (topic -1) above 20%? If yes, go back and **lower `hdbscan_min_cluster_size`** or **increase `umap_n_neighbors`** before continuing.
- Are important themes missing? Consider lowering `min_topic_size`.

Only proceed to notebook 02 when topics are interpretable.

## Save Outputs

In [ ]:
import numpy as np

# Save models
alt_model_dir = OUTPUT_DIR / "alt_media_model"
ms_model_dir = OUTPUT_DIR / "mainstream_model"

alt_topic_model.save(
    alt_model_dir,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=EMBEDDING_MODEL,
)
print(f"Saved alt media model:  {alt_model_dir}")

ms_topic_model.save(
    ms_model_dir,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=EMBEDDING_MODEL,
)
print(f"Saved mainstream model: {ms_model_dir}")

# Save topic info tables
alt_topic_info.to_csv(OUTPUT_DIR / "alt_media_topic_info.csv", index=False)
ms_topic_info.to_csv(OUTPUT_DIR / "mainstream_topic_info.csv", index=False)

# Save topic-document assignments
alt_doc_topics = alt_prepared[["document_id", "outlet_label", "document"]].copy()
alt_doc_topics["topic"] = alt_topics
alt_doc_topics.to_csv(OUTPUT_DIR / "alt_media_doc_topics.csv", index=False)

ms_doc_topics = mainstream_prepared[["document_id", "outlet_label", "document"]].copy()
ms_doc_topics["topic"] = ms_topics
ms_doc_topics.to_csv(OUTPUT_DIR / "mainstream_doc_topics.csv", index=False)

# Save topic embeddings (centroid vectors)
alt_embeddings = alt_topic_model._extract_embeddings(
    [" ".join(w for w, _ in alt_topic_model.get_topic(t)) for t in alt_topic_info["Topic"] if t != -1],
    method="document",
)
np.save(OUTPUT_DIR / "alt_media_topic_embeddings.npy", alt_embeddings)

ms_embeddings = ms_topic_model._extract_embeddings(
    [" ".join(w for w, _ in ms_topic_model.get_topic(t)) for t in ms_topic_info["Topic"] if t != -1],
    method="document",
)
np.save(OUTPUT_DIR / "mainstream_topic_embeddings.npy", ms_embeddings)

# Save topic word lists
import json

alt_word_lists = {
    str(t): [w for w, _ in alt_topic_model.get_topic(t)]
    for t in alt_topic_info["Topic"] if t != -1
}
ms_word_lists = {
    str(t): [w for w, _ in ms_topic_model.get_topic(t)]
    for t in ms_topic_info["Topic"] if t != -1
}

with open(OUTPUT_DIR / "alt_media_topic_words.json", "w") as f:
    json.dump(alt_word_lists, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "mainstream_topic_words.json", "w") as f:
    json.dump(ms_word_lists, f, ensure_ascii=False, indent=2)

print(f"\nAll outputs saved to: {OUTPUT_DIR}")